In [2]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
from datasets import load_dataset, Dataset

hf_dataset = load_dataset(
    "LeMaterial/LeMat-Bulk-v2-Test", 
    "compatible_pbe", 
    split="train", 
    num_proc=8,
    cache_dir="../LeMat-Bulk-v2-Test", # Adjust this path as needed
)
# 2. Add precomputed cost columns
cost_columns = np.load("data/cost_columns_only.npz", allow_pickle=True)

for col_name in ["nbands", "n_irr_kpts", "cost"]:
    hf_dataset = hf_dataset.add_column(col_name, cost_columns[col_name])

# 3. Remove 1D/2D structures known from alexandria
alex_low_dimensional = np.load("data/alex_2d_1d_ids.npz", allow_pickle=True)["immutable_ids"]
mask_keep = ~np.isin(hf_dataset["immutable_id"], alex_low_dimensional)
hf_dataset = hf_dataset.select(np.nonzero(mask_keep)[0])

table = pa.Table.from_batches(list(hf_dataset.data.to_batches()))

alexandria = pd.read_parquet('data/alex.parquet')
e_hull_df = alexandria[['mat_id', 'e_above_hull']]
e_hull_map = dict(zip(e_hull_df['mat_id'], e_hull_df['e_above_hull']))

#filter alexandria IDs
mask_alex = pc.is_in(table["immutable_id"], value_set=pa.array(e_hull_df['mat_id']))
table = table.filter(mask_alex)

ids = table["immutable_id"].to_numpy(zero_copy_only=False)
e_hull_values = np.array([e_hull_map.get(i, np.nan) for i in ids])
table = table.append_column("e_hull", pa.array(e_hull_values))

# Remove null nbands (some structures where missing columns so the nbands column is null)
table = table.filter(pc.invert(pc.is_null(table["nbands"])))

# Sort by nbands
sort_idx = pc.sort_indices(table, sort_keys=[("nbands", "ascending")])
table = pc.take(table, sort_idx)

#hf_dataset = Dataset(table)


Loaded dataset with 5370198 entries.


In [3]:
# Filter for e_hull < 0.05
mask_low = pc.less(table["e_hull"], pa.scalar(0.05))
table_low = table.filter(mask_low)

# Filter for 0.05 <= e_hull < 0.2
mask_mid = pc.and_(
    pc.greater_equal(table["e_hull"], pa.scalar(0.05)),
    pc.less(table["e_hull"], pa.scalar(0.2))
)
table_mid = table.filter(mask_mid)


In [4]:

def create_band_batches(table):
    """
    Create batches of datasets based on nbands ranges.
    Returns a dictionary mapping bin labels to Dataset objects.
    """
    nbands_values = table["nbands"].to_numpy(zero_copy_only=False).astype(float)

    #bin edges (example: 0–12, 12–24, ..., 288–300)
    bin_edges = np.arange(12, 300 + 12, 12) 
    bin_labels = [f"{start}-{end}" for start, end in zip(bin_edges[:-1], bin_edges[1:])]

    band_batches = {}

    for start, end, label in zip(bin_edges[:-1], bin_edges[1:], bin_labels):
        mask = pc.and_(
            pc.greater_equal(table["nbands"], pa.scalar(start)),
            pc.less(table["nbands"], pa.scalar(end))
        )
        batch_table = table.filter(mask)

        if batch_table.num_rows > 0:
            band_batches[label] = Dataset(batch_table)

    return band_batches

band_batches_low = create_band_batches(table_low)
band_batches_mid = create_band_batches(table_mid)

In [5]:
low_percentage = {label: len(dataset)/len(table_low) for label, dataset in band_batches_low.items()}
mid_percentage = {label: len(dataset)/len(table_mid) for label, dataset in band_batches_mid.items()}
cumulative_counts_low_percentage = 0
cumulative_counts_mid_percentage = 0
for label, dataset in band_batches_low.items():
    cumulative_counts_low_percentage += low_percentage[label]
    print(f"Ehull <50 meV/atom {label}: {len(dataset)} samples (cumulative: {cumulative_counts_low_percentage:.3%})")

for label, dataset in band_batches_mid.items():
    cumulative_counts_mid_percentage += mid_percentage[label]
    print(f"Ehull <50 meV/atom {label}: {len(dataset)} samples (cumulative: {cumulative_counts_mid_percentage:.3%})")

Ehull <50 meV/atom 12-24: 20368 samples (cumulative: 1.898%)
Ehull <50 meV/atom 24-36: 78040 samples (cumulative: 9.168%)
Ehull <50 meV/atom 36-48: 115461 samples (cumulative: 19.925%)
Ehull <50 meV/atom 48-60: 139487 samples (cumulative: 32.921%)
Ehull <50 meV/atom 60-72: 135834 samples (cumulative: 45.576%)
Ehull <50 meV/atom 72-84: 123565 samples (cumulative: 57.088%)
Ehull <50 meV/atom 84-96: 97211 samples (cumulative: 66.145%)
Ehull <50 meV/atom 96-108: 83481 samples (cumulative: 73.922%)
Ehull <50 meV/atom 108-120: 60872 samples (cumulative: 79.593%)
Ehull <50 meV/atom 120-132: 43030 samples (cumulative: 83.602%)
Ehull <50 meV/atom 132-144: 33683 samples (cumulative: 86.740%)
Ehull <50 meV/atom 144-156: 25537 samples (cumulative: 89.119%)
Ehull <50 meV/atom 156-168: 17529 samples (cumulative: 90.753%)
Ehull <50 meV/atom 168-180: 14285 samples (cumulative: 92.083%)
Ehull <50 meV/atom 180-192: 12678 samples (cumulative: 93.265%)
Ehull <50 meV/atom 192-204: 10120 samples (cumulative

In [6]:
META_DATA_KEYS = ['elements',
 'nsites',
 'chemical_formula_anonymous',
 'chemical_formula_reduced',
 'nperiodic_dimensions',
 'lattice_vectors',
 'immutable_id',
 'last_modified',
 'stress_tensor',
 'energy',
 'energy_corrected',
 'forces',
 'total_magnetization',
 'charges',
 'dos_ef',
 'functional',
 'cross_compatibility',
 'bawl_fingerprint',
 'space_group_it_number',
 'nbands',
 'n_irr_kpts',
 'cost']

from pymatgen.core import Structure

def get_structure_from_hf_row(row):
    """Get a pymatgen Structure from a dictionary.
    The dictionary should contain the following keys:
        - lattice_vectors: list of lists containing the lattice vectors
        - species_at_sites: list of species at each site
        - cartesian_site_positions: list of cartesian site positions

    Parameters
    ----------
    row : dict
        Dictionary containing the structure information.

    Returns
    -------
    pymatgen.Structure
        Pymatgen Structure object.
    """

    return Structure(
        lattice=[x for y in row["lattice_vectors"] for x in y],
        species=row["species_at_sites"],
        coords=row["cartesian_site_positions"],
        coords_are_cartesian=True,
    )


def structures_hf_dataset(dataset, num_proc=4):
    with ProcessPoolExecutor(max_workers=num_proc) as executor:
        structures = list(executor.map(get_structure_from_hf_row, dataset))
    return structures

META_DATA_KEYS = ['elements',
 'nsites',
 'chemical_formula_anonymous',
 'chemical_formula_reduced',
 'nperiodic_dimensions',
 'lattice_vectors',
 'immutable_id',
 'last_modified',
 'stress_tensor',
 'energy',
 'energy_corrected',
 'forces',
 'total_magnetization',
 'charges',
 'dos_ef',
 'functional',
 'cross_compatibility',
 'bawl_fingerprint',
 'space_group_it_number',
 'nbands',
 'n_irr_kpts',
 'cost']

from pymatgen.core import Structure
def get_structure_from_hf_row(row):
    """Get a pymatgen Structure from a dictionary.
    The dictionary should contain the following keys:
        - lattice_vectors: list of lists containing the lattice vectors
        - species_at_sites: list of species at each site
        - cartesian_site_positions: list of cartesian site positions

    Parameters
    ----------
    row : dict
        Dictionary containing the structure information.

    Returns
    -------
    pymatgen.Structure
        Pymatgen Structure object.
    """

    return Structure(
        lattice=[x for y in row["lattice_vectors"] for x in y],
        species=row["species_at_sites"],
        coords=row["cartesian_site_positions"],
        coords_are_cartesian=True,
    )


def metadata_list_hf_dataset(dataset):
    """Extract metadata from Hugging Face dataset in one vectorized pass."""
    # Pull all required columns at once
    columns = {key: dataset[key] for key in META_DATA_KEYS}
    # Build list of dicts by zipping values together
    return [dict(zip(META_DATA_KEYS, values)) for values in zip(*columns.values())]

def _build_structures_worker(args):
    lattice_vectors, species, positions, start, chunk_size = args
    out = []
    end = min(start + chunk_size, len(lattice_vectors))
    for i in range(start, end):
        out.append(Structure(
            lattice=[x for y in lattice_vectors[i] for x in y],
            species=species[i],
            coords=positions[i],
            coords_are_cartesian=True,
        ))
    return out

from concurrent.futures import ProcessPoolExecutor
def structures_hf_dataset(dataset, num_proc=4, chunk_size=1000):
    """Build pymatgen Structures from a Hugging Face dataset efficiently."""
    lattice_vectors = dataset["lattice_vectors"]
    species = dataset["species_at_sites"]
    positions = dataset["cartesian_site_positions"]

    # Prepare arguments for each chunk
    args_iter = [
        (lattice_vectors, species, positions, start, chunk_size)
        for start in range(0, len(lattice_vectors), chunk_size)
    ]

    # Parallelize over chunks
    with ProcessPoolExecutor(max_workers=num_proc) as executor:
        chunks = executor.map(_build_structures_worker, args_iter)

    # Flatten list of lists
    return [s for chunk in chunks for s in chunk]

batch_metadata_general = {"Description": "each minibatch 100 random structures from hf_24, increasing different cost intervals","BANDS24": 1, "Phase": "test"}

In [7]:
batch = band_batches_low["24-36"]
metadata_list = metadata_list_hf_dataset(batch)
print(f"Extracted {len(metadata_list)} metadata entries")
structures = structures_hf_dataset(batch, num_proc=4)
print(f"Built {len(structures)} structures")

Extracted 78040 metadata entries
Built 78040 structures


In [33]:
#test id-to-everything mapping is still correct
# import random
from pymatgen.core import Composition
for label, dataset in band_batches_low.items():
    random_entry = random.choice(dataset)
    immutable_id = random_entry["immutable_id"]
    alex_entry = alexandria[alexandria["mat_id"] == immutable_id]
    assert random_entry["e_hull"] == alex_entry["e_above_hull"].values[0]
    assert random_entry["energy"] == alex_entry["energy_total"].values[0]
    assert random_entry["nbands"] == alex_entry["nbands"].values[0]
    assert Composition(random_entry["chemical_formula_reduced"]) == Composition(alex_entry["formula"].values[0])

for label, dataset in band_batches_mid.items():
    random_entry = random.choice(dataset)
    immutable_id = random_entry["immutable_id"]
    alex_entry = alexandria[alexandria["mat_id"] == immutable_id]
    assert random_entry["e_hull"] == alex_entry["e_above_hull"].values[0]
    assert random_entry["energy"] == alex_entry["energy_total"].values[0]
    assert random_entry["nbands"] == alex_entry["nbands"].values[0]
    assert Composition(random_entry["chemical_formula_reduced"]) == Composition(alex_entry["formula"].values[0])
